# Vietnamese TTS Studio on Kaggle

Notebook nay dung cho Kaggle de:

1. Clone hoac pull repository tu GitHub.
2. Cai dependencies cho webapp TTS.
3. Chay Flask app tren Kaggle.
4. Mo Cloudflare tunnel de lay URL public.

Ban notebook nay da cat gon chi de lai engine chu dao `Gwen-TTS` de install nhanh va on dinh hon tren Kaggle.

Mac dinh notebook da tro san den repo hien tai:
`https://github.com/sin0235/text2speech.git`.

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/sin0235/text2speech.git"
BRANCH = "main"
WORKSPACE = Path("/kaggle/working")  # Kaggle working directory
PROJECT_DIR = WORKSPACE / Path(REPO_URL).stem.replace('.git', '')
PORT = 8386

INSTALL_GWEN = True
# qwen-tts==0.1.1 requires transformers==4.57.3.
# VibeVoice-ASR needs transformers>=5.3.0, so do not enable both in one runtime.
INSTALL_VIBEVOICE_ASR = False
DEFAULT_ENGINE = "gwen"
ENABLED_ENGINES = "gwen"

GWEN_MODEL_ID = "g-group-ai-lab/gwen-tts-0.6B"
GWEN_DTYPE = "bfloat16"
GWEN_ATTN_IMPLEMENTATION = "sdpa"
VIBEVOICE_ASR_MODEL_ID = "microsoft/VibeVoice-ASR-HF"
HF_TOKEN = ""

TORCH_VERSION = "2.11.0"
TORCH_SPEC = f"torch=={TORCH_VERSION}"
TORCHAUDIO_SPEC = f"torchaudio=={TORCH_VERSION}"
PYTORCH_GPU_INDEX = "https://download.pytorch.org/whl/cu128"
PYTORCH_CPU_INDEX = "https://download.pytorch.org/whl/cpu"

QWEN_TTS_PIP_SPEC = "qwen-tts==0.1.1"
QWEN_RUNTIME_DEPENDENCIES = [
    "transformers==4.57.3",
    "accelerate==1.12.0",
    "sentencepiece",
    "sox",
    "onnxruntime",
    "einops",
    "sea-g2p",
]

print({
    "repo_url": REPO_URL,
    "branch": BRANCH,
    "project_dir": str(PROJECT_DIR),
    "port": PORT,
    "install_gwen": INSTALL_GWEN,
    "install_vibevoice_asr": INSTALL_VIBEVOICE_ASR,
    "tts_enable_asr": int(INSTALL_VIBEVOICE_ASR),
    "default_engine": DEFAULT_ENGINE,
    "enabled_engines": ENABLED_ENGINES,
    "gwen_model_id": GWEN_MODEL_ID,
    "gwen_attn_implementation": GWEN_ATTN_IMPLEMENTATION,
    "qwen_tts_pip_spec": QWEN_TTS_PIP_SPEC,
    "torch_spec": TORCH_SPEC,
    "torchaudio_spec": TORCHAUDIO_SPEC,
    "pytorch_gpu_index": PYTORCH_GPU_INDEX,
})

{'repo_url': 'https://github.com/sin0235/text2speech.git', 'branch': 'main', 'project_dir': '/kaggle/working/text2speech', 'port': 8386, 'install_gwen': True, 'install_vibevoice_asr': False, 'tts_enable_asr': 0, 'default_engine': 'gwen', 'enabled_engines': 'gwen', 'gwen_model_id': 'g-group-ai-lab/gwen-tts-0.6B', 'gwen_attn_implementation': 'sdpa', 'qwen_tts_pip_spec': 'qwen-tts==0.1.1', 'torch_spec': 'torch==2.11.0', 'torchaudio_spec': 'torchaudio==2.11.0', 'pytorch_gpu_index': 'https://download.pytorch.org/whl/cu128'}


## Data Backup & Restore

Cell nay cai dat backup/restore du lieu:
- Luu lai output audio va cai dat phat am trong `/kaggle/working/tts_backup/`.
- Kaggle tu dong persist `/kaggle/working/` qua output, nen du lieu khong bi mat.
- Tu dong restore du lieu cu khi bat lai session.
- Tu dong sync truoc khi ket thuc session.


In [ ]:
import shutil
from pathlib import Path

KAGGLE_BACKUP_DIR = Path("/kaggle/working/tts_backup")
KAGGLE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
print(f"Backup directory: {KAGGLE_BACKUP_DIR}")

# Quick restore if backup exists from previous run
def _quick_restore():
    backup_outputs = KAGGLE_BACKUP_DIR / "outputs"
    if backup_outputs.exists() and any(backup_outputs.iterdir()):
        local_outputs = PROJECT_DIR / "outputs"
        local_outputs.mkdir(parents=True, exist_ok=True)
        count = 0
        for f in backup_outputs.iterdir():
            dest = local_outputs / f.name
            if not dest.exists():
                shutil.copy2(f, dest)
                count += 1
        print(f"Restored {count} output files from previous session.")
    backup_data = KAGGLE_BACKUP_DIR / "data"
    if backup_data.exists() and any(backup_data.iterdir()):
        local_data = PROJECT_DIR / "webapp" / "data"
        local_data.mkdir(parents=True, exist_ok=True)
        count = 0
        for f in backup_data.rglob("*"):
            if f.is_file():
                rel = f.relative_to(backup_data)
                dest = local_data / rel
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.copy2(f, dest)
                    count += 1
        print(f"Restored {count} data files from previous session.")

_quick_restore()


In [2]:
import subprocess

WORKSPACE.mkdir(parents=True, exist_ok=True)

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

subprocess.run(["git", "-C", str(PROJECT_DIR), "status", "--short"], check=True)
print(f"Project ready at: {PROJECT_DIR}")

Project ready at: /kaggle/working/text2speech


In [3]:
import importlib.metadata as importlib_metadata
import shutil
import site
import subprocess
import sys
from pathlib import Path

subprocess.run(["apt-get", "update", "-y"], check=True)
subprocess.run(["apt-get", "install", "-y", "ffmpeg", "libsndfile1", "espeak-ng"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"], check=True)

has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
pytorch_index = PYTORCH_GPU_INDEX if has_gpu else PYTORCH_CPU_INDEX
print({"has_gpu": has_gpu, "pytorch_index": pytorch_index, "torch_spec": TORCH_SPEC, "torchaudio_spec": TORCHAUDIO_SPEC})

TORCH_STACK_PACKAGES = [
    "torch",
    "torchaudio",
]
RESIDUE_PATTERNS = [
    "torch",
    "torch-*",
    "torchaudio",
    "torchaudio-*",
    "functorch",
    "torchgen",
]

def _site_package_roots():
    roots = []
    for raw in [*site.getsitepackages(), site.getusersitepackages()]:
        if not raw:
            continue
        root = Path(raw)
        if root.exists() and root not in roots:
            roots.append(root)
    return roots

def _purge_residue(patterns):
    for root in _site_package_roots():
        for pattern in patterns:
            for path in root.glob(pattern):
                try:
                    if path.is_dir():
                        shutil.rmtree(path, ignore_errors=True)
                    elif path.exists():
                        path.unlink(missing_ok=True)
                except FileNotFoundError:
                    pass

def _run_pip_install(*packages, force_reinstall=False, no_deps=False):
    cmd = [sys.executable, "-m", "pip", "install", "--upgrade", "--no-cache-dir"]
    if force_reinstall:
        cmd.append("--force-reinstall")
    if no_deps:
        cmd.append("--no-deps")
    cmd.extend(packages)
    subprocess.run(cmd, check=True)

def _install_torch_stack(force_reinstall=True):
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", *TORCH_STACK_PACKAGES], check=False)
    _purge_residue(RESIDUE_PATTERNS)
    cmd = [
        sys.executable, "-m", "pip", "install",
        "--upgrade",
        "--no-cache-dir",
    ]
    if force_reinstall:
        cmd.append("--force-reinstall")
    cmd.extend([
        TORCH_SPEC,
        TORCHAUDIO_SPEC,
        "--index-url", pytorch_index,
    ])
    subprocess.run(cmd, check=True)

def _verify_torch_stack(label="Torch stack"):
    verify_code = "\n".join([
        "import json",
        "import torch",
        "import torchaudio",
        "print(json.dumps({",
        "    'torch': torch.__version__,",
        "    'torchaudio': torchaudio.__version__,",
        "    'cuda_available': torch.cuda.is_available(),",
        "}))",
    ])
    completed = subprocess.run(
        [sys.executable, "-c", verify_code],
        check=False,
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        print(f"{label} verify failed.")
        if completed.stdout:
            print(completed.stdout.strip())
        if completed.stderr:
            print(completed.stderr.strip())
        raise subprocess.CalledProcessError(
            completed.returncode,
            completed.args,
            output=completed.stdout,
            stderr=completed.stderr,
        )
    print(f"{label}: {completed.stdout.strip()}")

def _install_qwen_runtime(force_reinstall=False):
    # Install only the runtime needed by the Flask TTS app.
    # `qwen-tts` pulls `gradio` by default, but this notebook does not use the demo UI.
    _run_pip_install(*QWEN_RUNTIME_DEPENDENCIES, force_reinstall=force_reinstall)
    _run_pip_install(QWEN_TTS_PIP_SPEC, force_reinstall=force_reinstall, no_deps=True)
    if has_gpu and GWEN_ATTN_IMPLEMENTATION == "flash_attention_2":
        try:
            _run_pip_install("--no-build-isolation", "flash-attn")
        except subprocess.CalledProcessError as exc:
            globals()["GWEN_ATTN_IMPLEMENTATION"] = "sdpa"
            print(f"flash-attn install failed, fallback to sdpa: {exc}")

def _verify_qwen_runtime(label="Qwen runtime"):
    verify_code = "\n".join([
        "import importlib.metadata as md",
        "import json",
        "import qwen_tts",
        "import sea_g2p",
        "print(json.dumps({",
        "    'qwen_tts': md.version('qwen-tts'),",
        "    'transformers': md.version('transformers'),",
        "    'sea_g2p': md.version('sea-g2p'),",
        "}))",
    ])
    completed = subprocess.run(
        [sys.executable, "-c", verify_code],
        check=False,
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        print(f"{label} verify failed.")
        if completed.stdout:
            print(completed.stdout.strip())
        if completed.stderr:
            print(completed.stderr.strip())
        raise subprocess.CalledProcessError(
            completed.returncode,
            completed.args,
            output=completed.stdout,
            stderr=completed.stderr,
        )
    print(f"{label}: {completed.stdout.strip()}")

_install_torch_stack()
try:
    _verify_torch_stack("Torch stack after base install")
except subprocess.CalledProcessError as exc:
    print("Torch stack verify failed once, retrying after deeper cleanup.")
    if exc.stdout:
        print(exc.stdout.strip())
    if exc.stderr:
        print(exc.stderr.strip())
    _install_torch_stack()
    _verify_torch_stack("Torch stack after base reinstall")

_run_pip_install("-r", str(PROJECT_DIR / "requirements.txt"), "huggingface_hub")

if INSTALL_VIBEVOICE_ASR and INSTALL_GWEN:
    print("XUNG DOT: qwen-tts can transformers==4.57.3, VibeVoice-ASR can transformers>=5.3.0")
    print("Notebook nay giu TTS-only, bo qua VibeVoice-ASR de tranh download dependency khong can thiet.")
    INSTALL_VIBEVOICE_ASR = False

if INSTALL_GWEN:
    _install_qwen_runtime()
    try:
        _verify_qwen_runtime("Qwen runtime after install")
    except subprocess.CalledProcessError:
        print("Qwen runtime verify failed, reinstalling exact pinned runtime without demo dependencies.")
        _install_qwen_runtime(force_reinstall=True)
        _install_torch_stack()
        _verify_torch_stack("Torch stack after qwen runtime repair")
        _verify_qwen_runtime("Qwen runtime after repair")

try:
    _verify_torch_stack("Torch stack after optional installs")
except subprocess.CalledProcessError:
    print("Optional packages changed the PyTorch stack. Reinstalling pinned torch/torchaudio.")
    _install_torch_stack()
    _verify_torch_stack("Torch stack after final repair")

if INSTALL_GWEN:
    print({
        "qwen_tts": importlib_metadata.version("qwen-tts"),
        "transformers": importlib_metadata.version("transformers"),
        "sea_g2p": importlib_metadata.version("sea-g2p"),
        "gwen_model_id": GWEN_MODEL_ID,
        "gwen_attn_implementation": GWEN_ATTN_IMPLEMENTATION,
    })

print("Notebook is configured for TTS-only runtime; model weights are not pre-downloaded.")
print(f"Dependencies installed for enabled engines: {ENABLED_ENGINES}")

{'has_gpu': True, 'pytorch_index': 'https://download.pytorch.org/whl/cu128', 'torch_spec': 'torch==2.11.0', 'torchaudio_spec': 'torchaudio==2.11.0'}


KeyboardInterrupt: 

In [ ]:
import os

RUNTIME_ENV = os.environ.copy()
RUNTIME_ENV["TTS_DEFAULT_ENGINE"] = DEFAULT_ENGINE
RUNTIME_ENV["TTS_ENABLED_ENGINES"] = ENABLED_ENGINES
RUNTIME_ENV["TTS_ENABLE_ASR"] = "1" if INSTALL_VIBEVOICE_ASR else "0"
RUNTIME_ENV["GWEN_MODEL_ID"] = GWEN_MODEL_ID
RUNTIME_ENV["GWEN_DTYPE"] = GWEN_DTYPE
RUNTIME_ENV["GWEN_ATTN_IMPLEMENTATION"] = GWEN_ATTN_IMPLEMENTATION
if INSTALL_VIBEVOICE_ASR:
    RUNTIME_ENV["VIBEVOICE_ASR_MODEL_ID"] = VIBEVOICE_ASR_MODEL_ID

if HF_TOKEN.strip():
    from huggingface_hub import login
    login(HF_TOKEN)
    print("Hugging Face token configured.")
else:
    print("HF_TOKEN is empty. Public downloads only.")

keys = [
    "TTS_DEFAULT_ENGINE",
    "TTS_ENABLED_ENGINES",
    "TTS_ENABLE_ASR",
    "GWEN_MODEL_ID",
    "GWEN_DTYPE",
    "GWEN_ATTN_IMPLEMENTATION",
]
if INSTALL_VIBEVOICE_ASR:
    keys.append("VIBEVOICE_ASR_MODEL_ID")
print({k: RUNTIME_ENV[k] for k in keys})

In [ ]:
import os
import signal
import subprocess
import sys
import time
import urllib.request
from pathlib import Path
from IPython.display import clear_output

APP_LOG = PROJECT_DIR / "webapp_kaggle.log"
APP_PID = PROJECT_DIR / "webapp_kaggle.pid"
TUNNEL_LOG = PROJECT_DIR / "cloudflared.log"
TUNNEL_PID = PROJECT_DIR / "cloudflared.pid"

def _pid_alive(pid: int) -> bool:
    try:
        os.kill(pid, 0)
        return True
    except Exception:
        return False

def _process_status(pid_file: Path, label: str) -> str:
    if not pid_file.exists():
        return f"{label}: chua co PID"
    try:
        pid = int(pid_file.read_text().strip())
    except Exception:
        return f"{label}: PID file khong hop le"
    state = "dang chay" if _pid_alive(pid) else "da thoat"
    return f"{label}: PID {pid} {state}"

def _read_log_tail(path: Path, *, lines: int = 120, max_chars: int = 12000) -> str:
    if not path.exists():
        return "(log file chua ton tai)"
    raw = path.read_text(encoding="utf-8", errors="ignore")
    if not raw.strip():
        return "(log file dang rong)"
    parts = raw.splitlines()
    clipped = "\n".join(parts[-max(1, int(lines)):])
    return clipped[-max_chars:]

def show_backend_logs(lines: int = 120, include_cloudflare: bool = False, max_chars: int = 12000) -> None:
    print("=== Trang thai process ===")
    print(_process_status(APP_PID, "Flask"))
    if include_cloudflare:
        print(_process_status(TUNNEL_PID, "Cloudflare"))
    print("\n=== Flask log ===")
    print(_read_log_tail(APP_LOG, lines=lines, max_chars=max_chars))
    if include_cloudflare:
        print("\n=== Cloudflare log ===")
        print(_read_log_tail(TUNNEL_LOG, lines=lines, max_chars=max_chars))

def watch_backend_logs(seconds=None, interval: int = 2, lines: int = 120, include_cloudflare: bool = False, max_chars: int = 12000) -> None:
    end_at = None if seconds is None else time.time() + max(1, int(seconds))
    tick = max(1, int(interval))
    try:
        while True:
            clear_output(wait=True)
            if end_at is None:
                print("Dang theo doi backend log lien tuc. Interrupt cell de dung.")
            else:
                remaining = max(0, int(end_at - time.time()))
                print(f"Dang theo doi backend log... con {remaining}s. Interrupt cell de dung som.")
            print()
            show_backend_logs(lines=lines, include_cloudflare=include_cloudflare, max_chars=max_chars)
            if end_at is not None and time.time() >= end_at:
                break
            time.sleep(tick)
    except KeyboardInterrupt:
        print("Da dung theo doi backend log.")

if APP_PID.exists():
    try:
        old_pid = int(APP_PID.read_text().strip())
        os.kill(old_pid, signal.SIGTERM)
        time.sleep(2)
        if _pid_alive(old_pid):
            os.kill(old_pid, signal.SIGKILL)
    except Exception:
        pass

subprocess.run(["bash", "-lc", f"fuser -k {PORT}/tcp || true"], check=False)
RUNTIME_ENV["PYTHONUNBUFFERED"] = "1"
launch_cmd = [
    sys.executable,
    "-u",
    "-c",
    (
        "from webapp.app import app; "
        f"app.run(host='0.0.0.0', port={PORT}, debug=False, use_reloader=False)"
    ),
]

log_file = APP_LOG.open("w", encoding="utf-8")
app_proc = subprocess.Popen(
    launch_cmd,
    cwd=str(PROJECT_DIR),
    env=RUNTIME_ENV,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
    start_new_session=True,
)
log_file.close()

APP_PID.write_text(str(app_proc.pid), encoding="utf-8")
status_url = f"http://127.0.0.1:{PORT}/api/tts/status"
ready = False

for _ in range(60):
    if app_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen(status_url, timeout=2) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        pass
    time.sleep(1)

log_tail = _read_log_tail(APP_LOG, lines=160, max_chars=12000)
if not ready:
    raise RuntimeError(
        f"Flask khong len duoc hoac da thoat som. Return code: {app_proc.poll()}\n\n{log_tail}"
    )

print(f"Flask PID: {app_proc.pid}")
print(f"Status URL OK: {status_url}")
print("Da define show_backend_logs() va watch_backend_logs() de xem log backend ngay trong notebook.")
print("Goi show_backend_logs() de xem nhanh, hoac watch_backend_logs(interval=2, include_cloudflare=True) de theo doi lien tuc.")
print("Neu muon tu dung sau mot khoang thoi gian, dung watch_backend_logs(seconds=60, interval=2, include_cloudflare=True).")
print()
show_backend_logs(lines=120)

In [ ]:
from pathlib import Path
import stat
import subprocess
import urllib.request

CLOUDFLARED_BIN = Path("/usr/local/bin/cloudflared")
CLOUDFLARED_URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"

if not CLOUDFLARED_BIN.exists():
    urllib.request.urlretrieve(CLOUDFLARED_URL, CLOUDFLARED_BIN)
    CLOUDFLARED_BIN.chmod(CLOUDFLARED_BIN.stat().st_mode | stat.S_IEXEC)

subprocess.run([str(CLOUDFLARED_BIN), "--version"], check=True)
print(f"cloudflared ready at {CLOUDFLARED_BIN}")

In [ ]:
import os
import re
import signal
import subprocess
import time
import urllib.request
from IPython.display import HTML, display

APP_PID = PROJECT_DIR / "webapp_kaggle.pid"
TUNNEL_LOG = PROJECT_DIR / "cloudflared.log"
TUNNEL_PID = PROJECT_DIR / "cloudflared.pid"

if not APP_PID.exists():
    raise RuntimeError("Chua co Flask PID. Hay chay cell start Flask truoc.")

app_pid = int(APP_PID.read_text().strip())
try:
    os.kill(app_pid, 0)
except Exception as exc:
    raise RuntimeError(f"Flask PID {app_pid} khong con song. Mo log webapp_kaggle.log de xem loi goc.") from exc

if TUNNEL_PID.exists():
    try:
        old_tunnel_pid = int(TUNNEL_PID.read_text().strip())
        os.kill(old_tunnel_pid, signal.SIGTERM)
        time.sleep(2)
        try:
            os.kill(old_tunnel_pid, 0)
            os.kill(old_tunnel_pid, signal.SIGKILL)
        except Exception:
            pass
    except Exception:
        pass

log_file = TUNNEL_LOG.open("w", encoding="utf-8")
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    cwd=str(PROJECT_DIR),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
    start_new_session=True,
)
log_file.close()

TUNNEL_PID.write_text(str(tunnel_proc.pid), encoding="utf-8")
pattern = re.compile(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com")
public_url = None

for _ in range(90):
    if tunnel_proc.poll() is not None:
        break
    time.sleep(1)
    log_text = TUNNEL_LOG.read_text(encoding='utf-8', errors='ignore') if TUNNEL_LOG.exists() else ""
    match = pattern.search(log_text)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError(
        "Khong tim thay trycloudflare URL. Kiem tra cloudflared.log.\n\n"
        + TUNNEL_LOG.read_text(encoding='utf-8', errors='ignore')[-4000:]
    )

for _ in range(15):
    try:
        with urllib.request.urlopen(public_url, timeout=10) as resp:
            if resp.status < 500:
                break
    except Exception:
        time.sleep(1)

print("Public URL:", public_url)
display(HTML(f'<a href="{public_url}" target="_blank">Open TTS webapp</a>'))

In [ ]:
# Xem log backend/cloudflare lien tuc. Interrupt cell khi muon dung.
watch_backend_logs(interval=2, lines=140, include_cloudflare=True)

In [ ]:
import os
import signal
import shutil
import time

KAGGLE_BACKUP_DIR = Path("/kaggle/working/tts_backup")

def sync_to_kaggle_backup():
    """Sync outputs and data to Kaggle working directory for persistence."""
    KAGGLE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    synced = []
    local_outputs = PROJECT_DIR / "outputs"
    if local_outputs.exists():
        backup_outputs = KAGGLE_BACKUP_DIR / "outputs"
        backup_outputs.mkdir(parents=True, exist_ok=True)
        count = 0
        for f in local_outputs.iterdir():
            if f.is_file():
                dest = backup_outputs / f.name
                if not dest.exists() or f.stat().st_mtime > dest.stat().st_mtime:
                    shutil.copy2(f, dest)
                    count += 1
        if count:
            synced.append(f"outputs: {count} files")
    local_data = PROJECT_DIR / "webapp" / "data"
    if local_data.exists():
        backup_data = KAGGLE_BACKUP_DIR / "data"
        backup_data.mkdir(parents=True, exist_ok=True)
        count = 0
        for f in local_data.rglob("*"):
            if f.is_file():
                rel = f.relative_to(local_data)
                dest = backup_data / rel
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists() or f.stat().st_mtime > dest.stat().st_mtime:
                    shutil.copy2(f, dest)
                    count += 1
        if count:
            synced.append(f"data: {count} files")
    if synced:
        print(f"Synced to backup: {', '.join(synced)}")
    else:
        print("No new files to backup.")

def restore_from_kaggle_backup():
    """Restore outputs and data from Kaggle backup if available."""
    if not KAGGLE_BACKUP_DIR.exists():
        return
    restored = []
    backup_outputs = KAGGLE_BACKUP_DIR / "outputs"
    if backup_outputs.exists() and any(backup_outputs.iterdir()):
        local_outputs = PROJECT_DIR / "outputs"
        local_outputs.mkdir(parents=True, exist_ok=True)
        count = 0
        for f in backup_outputs.iterdir():
            dest = local_outputs / f.name
            if not dest.exists():
                shutil.copy2(f, dest)
                count += 1
        if count:
            restored.append(f"outputs: {count} files")
    backup_data = KAGGLE_BACKUP_DIR / "data"
    if backup_data.exists() and any(backup_data.iterdir()):
        local_data = PROJECT_DIR / "webapp" / "data"
        local_data.mkdir(parents=True, exist_ok=True)
        count = 0
        for f in backup_data.rglob("*"):
            if f.is_file():
                rel = f.relative_to(backup_data)
                dest = local_data / rel
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.copy2(f, dest)
                    count += 1
        if count:
            restored.append(f"data: {count} files")
    if restored:
        print(f"Restored from backup: {', '.join(restored)}")
    else:
        print("No previous backup found, starting fresh.")

def stop_services():
    # Sync before stopping
    try:
        print("Syncing data to backup before stopping...")
        sync_to_kaggle_backup()
    except Exception as exc:
        print(f"Backup sync failed: {exc}")
    # Stop processes
    for pid_file in [PROJECT_DIR / "webapp_kaggle.pid", PROJECT_DIR / "cloudflared.pid"]:
        if not pid_file.exists():
            continue
        try:
            pid = int(pid_file.read_text().strip())
            os.kill(pid, signal.SIGTERM)
            time.sleep(1)
            try:
                os.kill(pid, 0)
                os.kill(pid, signal.SIGKILL)
            except Exception:
                pass
            print(f"Stopped PID from {pid_file.name}")
        except Exception as exc:
            print(f"Could not stop {pid_file.name}: {exc}")

print("Da define stop_services(). Chi goi stop_services() khi ban muon tat Flask va Cloudflare tunnel.")
print("stop_services() se tu dong sync du lieu truoc khi tat.")
print("Du lieu duoc luu tai /kaggle/working/tts_backup/ (persist qua Kaggle output).")
